# ex07_lti_wts_mcs

Interactive notebook version of the Python example.
Edit parameters and rerun cells to explore the model behavior.


## MATLAB-style Sections

- LTI model of a Coleman transformed wind turbine system
- Closed-loop identification experiment
- Uncertainty bounds using Monte Carlo simulations


## Imports


In [5]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from _common import (
    estimate_varx_abcdk,
    frequency_response,
    simulate_lti,
    stable_random_system,
    state_space_model,
    vaf_percent,
)
from pbsid.plotting import dbodemagpatch, plot_pole_map

## LTI Model Of A Coleman Transformed Wind Turbine System
Generate a stable surrogate LTI wind turbine model for the Monte Carlo notebook port.

In [2]:
a, b, c, d, k = stable_random_system(n=7, r=2, l_out=3, seed=107)

## Closed-Loop Identification Experiment
Run the surrogate closed-loop Monte Carlo identification workflow.

In [6]:
h = 0.1
n_samples = 4000
mcs = 20
rng = np.random.default_rng(107)
w = np.logspace(-2, np.log10(np.pi / h), 1000)
reference_response = np.abs(frequency_response(state_space_model(a, b, c, d, h), w))
vaf_store = np.zeros((mcs, 3), dtype=np.float64)
pole_store = np.zeros((mcs, a.shape[0]), dtype=np.complex128)
response_min = None
response_max = None
response_best = None
best_mean_vaf = -np.inf
for i in range(mcs):
    u = np.column_stack(
        (
            np.sign(rng.standard_normal(n_samples)),
            1e3 * np.sign(rng.standard_normal(n_samples)),
        )
    ).astype(np.float64)
    y, _ = simulate_lti(a, b, c, d, u, k=k, e=0.03 * rng.standard_normal((n_samples, 3)))
    y_nom, _ = simulate_lti(a, b, c, d, u)
    _, ai, bi, ci, di, _, _, _ = estimate_varx_abcdk(u, y, n=a.shape[0], f=20, p=50)
    yi, _ = simulate_lti(ai, bi, ci, di, u)
    vaf_store[i, :] = vaf_percent(y_nom, yi)
    pole_store[i, :] = np.linalg.eigvals(ai)
    response = np.abs(frequency_response(state_space_model(ai, bi, ci, di, h), w))
    if response_min is None:
        response_min = response.copy()
        response_max = response.copy()
    else:
        response_min = np.minimum(response_min, response)
        response_max = np.maximum(response_max, response)
    mean_vaf = float(np.mean(vaf_store[i, :]))
    if response_best is None or mean_vaf > best_mean_vaf:
        best_mean_vaf = mean_vaf
        response_best = response.copy()

c:\Users\Atindriyo\anaconda3\envs\pbsid-py\Lib\site-packages\control\lti.py:138: UserWarning: __call__: evaluation above Nyquist frequency
  warn("__call__: evaluation above Nyquist frequency")


## Identification Results
Summarize the Monte Carlo VAF statistics and pole magnitudes.

In [ ]:
plot_pole_map(
    np.linalg.eigvals(a),
    [("Monte Carlo", pole_store, "x", "tab:blue")],
    "Monte Carlo pole cloud",
)

In [ ]:
plt.figure(figsize=(10, 7))
dbodemagpatch(response_best, response_min, response_max, w, h, reference_response)
plt.suptitle("Monte Carlo Bode magnitude envelope")
plt.tight_layout()

In [4]:
print("[ex07-monte-carlo]")
print(f"VAF mean (%): {np.array2string(np.mean(vaf_store, axis=0), precision=2)}")
print(f"VAF std  (%): {np.array2string(np.std(vaf_store, axis=0), precision=2)}")
print(
    f"Mean pole magnitudes: {np.array2string(np.mean(np.abs(pole_store), axis=0), precision=4)}"
)

[ex07-monte-carlo]
VAF mean (%): [100. 100. 100.]
VAF std  (%): [1.61e-09 4.58e-10 2.36e-10]
Mean pole magnitudes: [0.6439 0.8865 0.7654 0.8541 0.7994 0.8307 0.811 ]
